### 3.3.1 Convolutional Autoencoder (notebooks 02 and 03)

#### 3.3.1.1 Inspiration for our approach

Our dataset setup is inherently unsupervised for training: only normal cable images are available in the training set. Because of this, we adopted a reconstruction-based anomaly detection strategy with convolutional autoencoders. The core idea is to train a model that reconstructs normal structures well; at inference time, anomalous regions should reconstruct poorly and therefore produce higher reconstruction error.

From Notebook 02, the first 3-block baseline established this principle but also showed that a relatively large bottleneck can lead to near-identity reconstruction of some defects. Notebook 03 then explored progressively tighter bottlenecks (4-block, 5-block, and 6-block) to reduce defect memorization and improve anomaly separation.

#### 3.3.1.2 Baseline (3-block)

**Architecture (Notebook 02, `Autoencoder3Block`)**  
The baseline model is a symmetric 3-block convolutional autoencoder:
- Encoder: `Conv(3->16) -> Conv(16->32) -> Conv(32->64)`, each block followed by BatchNorm, ReLU, and `MaxPool2d(2)`.
- Bottleneck: `[B, 64, 32, 32]` for 256x256 input (about 33.3% latent retention relative to raw spatial input size).
- Decoder: mirrored transposed-convolution path `64->32->16->3` with BatchNorm + ReLU and final Sigmoid output.

**Model Training (Notebook 02)**
- Batch size: 16
- Optimizer: Adam
- Learning rate: `1e-3`
- Loss: combined L1 + SSIM loss,  
  $L = \lambda L_1 + (1-\lambda)(1-\mathrm{SSIM})$, with `\lambda = 0.5`
- Epoch schedule: 150 epochs, then extended by another 150 epochs (300 total)
- Early stopping: default patience in `train_autoencoder` (10)

**Loss Curve (Notebook 02 figure flow)**  

![img](../figures/Fig3_Baseline_Loss_150.png)
![img](../figures/Fig4_Baseline_Loss_300.png)
Following the notebook sequence, the first loss-curve plot (150 epochs) shows convergence with continued room for improvement in log scale; the extended 300-epoch curve then confirms further reduction after continuation. A visible transition spike at the extension boundary is consistent with optimizer state reset when starting the second run.

**Anomaly Map Visualisation**

![image](../figures/Fig8_Baseline_Anomaly_Maps.png)

#### 3.3.1.3 4-block

**Architecture (Notebook 03, `Autoencoder4Block`)**  
The 4-block variant deepens the encoder-decoder stack:
- Encoder channels: `3->16->32->64->128`
- Bottleneck: `[B, 128, 16, 16]` (about 16.7% latent retention)
- Decoder mirrors this with transposed convolutions back to `[B, 3, 256, 256]`.

**Model Training (Notebook 03)**
- Batch size: 16
- Optimizer: Adam
- Learning rate: `1e-3`
- Loss weighting: `L1_WEIGHT = 0.5`
- Max epochs: 350
- Early stopping: default patience (10)

**Loss Curve (Notebook 03 figure flow)**  

![img](../figures/Fig9_4Block_Loss.png)
In the corresponding 4-block loss-curve section, training is stable but validation loss plateaus higher than the 3-block baseline, reflecting stronger compression and reduced reconstruction capacity.

**Anomaly Map Visualisation**

![img](../figures/Fig13_4Block_Anomaly_Maps.png)

**Comparison** 
The evaluation metrics compared to our 3-block baseline:

- **Image AUROC: 0.4625** (increased from 0.4412)
    - The model's ability to classify entire images improved slightly, but performance remains poor, since the score is still below 0.5 which is worse than random guessing.
    - Looking at the anomaly maps, the model still appears to be struggling to reconstruct overexposed regions, like the shiny wire ends.
    - The magnitude of reconstruction errors seems to be higher than in the baseline (many more red spots in the heatmap).
- **Pixel AUROC: 0.6147** (increased from 0.5878)
    - This metric improved slightly, likely due to the increased magnitude of reconstruction errors around the overexposed wire ends, which improves scores for defect classes like `bent_wire` and `combined` that have overexposed areas in their defects, as seen from the heatmaps.
- **AUPRO: 0.2455** (increased from 0.1270)
    - Because the 4-block model is now producing much higher error magnitudes for overexposure in general, it inadvertently highlights some defect classes more strongly now. The inability of the model to reconstruct overexposed areas seems to contribute somewhat to the increased scores for both Pixel AUROC and AUPRO.

#### 3.3.1.4 5-block

**Architecture (Notebook 03, `Autoencoder5Block`)**  
The 5-block model further compresses the latent space:
- Encoder channels: `3->16->32->64->128->256`
- Bottleneck: `[B, 256, 8, 8]` (about 8.3% latent retention)
- Decoder is symmetric and reconstructs to full input resolution.

**Model Training (Notebook 03)**
- Batch size: 16
- Optimizer: Adam
- Learning rate: `1e-3`
- Loss weighting: `L1_WEIGHT = 0.5`
- Epochs: 200
- Early stopping patience: 20

**Loss Curve (Notebook 03 figure flow)**  
![img](../figures/Fig14_5Block_Loss.png)
The 5-block loss-curve section indicates continued convergence under tighter compression, and this configuration gave the best overall anomaly metrics among the pure convolutional autoencoder variants tested.

**Anomaly Map Visualisation**

![img](../figures/Fig16_5Block_Anomaly_Maps.png)

#### 3.3.1.5 6-block

**Architecture (Notebook 03, `Autoencoder6Block`)**  
The 6-block model uses the strongest compression among CAE variants:
- Encoder channels: `3->16->32->64->128->256->512`
- Bottleneck: `[B, 512, 4, 4]` (about 4.17% latent retention)
- Decoder mirrors the encoder depth to reconstruct 256x256 RGB output.

**Model Training (Notebook 03)**
- Batch size: 16
- Optimizer: Adam
- Learning rate: `1e-3`
- Loss weighting: `L1_WEIGHT = 0.5`
- Epochs: 100
- Early stopping patience: 20

**Loss Curve (Notebook 03 figure flow)**  
![img](../figures/Fig17_6Block_Loss.png)
The 6-block loss-curve section follows the same training-analysis structure and is used together with downstream evaluation to assess whether extreme bottleneck compression improves anomaly localization. In our experiments, it improved pixel-level behavior but did not improve all headline metrics, indicating diminishing returns from further compression alone.

**Anomaly Map Visualisation**

![img](../figures/Fig19_6Block_Anomaly_Maps.png)

**Results Summary (Notebooks 02 and 03)**
| Metric\Model | Baseline 3-Block | 4-Block | 5-Block | 6-block |
| :--- | :--- | :--- | :--- | :---
| **Image AUROC** | 0.4412 | 0.4625 | 0.5547 | 0.5380 |
| **Pixel AUROC** | 0.5878 | 0.6147 | 0.6937 | 0.7669 |
| **AUPRO** | 0.1270 | 0.2455 | 0.3625 | 0.3617 |

Across all models, progressively tightening the bottleneck improved all three evaluation metrics. The 3-block baseline performed the worst, suggesting that its latent space was too permissive and allowed many defects to be reconstructed too faithfully. The 4-block model showed only modest improvement, and as discussed previously, might indicate that a slightly tighter bottleneck was still insufficient to meaningfully suppress anomalous structures.

So we trained the 5-block model, which has **performed the best on all three metrics** - Image AUROC, Pixel AUROC, and AUPRO - so far. Compared to the 4-block model, the 5-block model was better at **failing to reconstruct certain defects like cut outer insulation and poke insulation**, as seen from the anomaly heatmaps above. However, bigger holes like missing cable were still being reconstructed. Still, **the 5-block model is an improvement over the 3 and 4-block models**, and it suggests that tightening the bottleneck works in preventing some anomalous structures from passing through the latent space.

However, just like the 3 and 4-block models, the 5-block model fails to reproduce the overexposed shiny wire ends, which remain a strong source of reconstruction error even in normal images. This likely helps explain why the Image AUROC, although improved, is still relatively low. The model is therefore responding to both true defects (some) and regions with difficult lighting.

This improvement also came with a clear trade-off. The 5-block reconstructions are noticeably less saturated, more blurred, and more grey overall than those of the 3-block and 4-block models. In other words, stronger compression reduces defect memorization, but also removes useful normal structure and makes the reconstructed images less accurate overall.

We then tried a 6-block model, however we notice that while there are improvements in the Pixel AUROC, the other two metrics got slightly worse.